# preTextAnalysis
Entrada: `acciones_seleccionadas.csv` (8 partidos, acciones ya seleccionadas a mano en "Economía y papel del Estado"). Salida: `acciones_preparadas.csv`, listo para `TextAnalysis`.

## preTEXT-01: leer el corpus
Lee el CSV directamente desde el repositorio del curso.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/doctorado-cuanticp/text/main/acciones_seleccionadas.csv"
corpus = pd.read_csv(URL, keep_default_na=False)
corpus

## preTEXT-02: estructura
Ocho filas, una por partido. Se registra `num_propuestas` (varía entre 4 y 9) porque afecta la interpretación en `TextAnalysis`.

In [ ]:
print('Total de partidos:', len(corpus))

corpus['num_propuestas'] = corpus['texto_original'].apply(
    lambda t: len([l for l in str(t).split('\n') if l.strip()])
)
corpus[['id_partido', 'partido', 'seccion', 'pagina', 'num_propuestas']]

## preTEXT-03: preparar el texto
Normaliza espacios y acentos; conserva negaciones, condiciones y puntuación.

In [ ]:
import re, unicodedata

def limpiar(texto):
    texto = unicodedata.normalize('NFC', texto)
    texto = texto.replace('\u00ad', '')
    return re.sub(r'\s+', ' ', texto).strip()

corpus['texto'] = corpus['texto_original'].apply(limpiar)
corpus[['id_partido', 'texto']]

## preTEXT-04: comparar con el original
Confirma que la limpieza no alteró contenido. No valida la selección manual — eso exige volver al PDF.

In [ ]:
for _, fila in corpus.iterrows():
    print(f"\n{fila['id_partido']} — {fila['partido']}")
    print('ORIGINAL :', fila['texto_original'][:150], '...')
    print('PREPARADO:', fila['texto'][:150], '...')

## preTEXT-05: guardar
Exporta el corpus final, una fila por partido.

In [ ]:
assert len(corpus) > 0 and corpus['texto'].ne('').all(), 'Complete los textos antes de guardar.'
assert corpus['id_partido'].is_unique, 'Debe haber una sola fila por partido.'

corpus.to_csv('acciones_preparadas.csv', index=False, encoding='utf-8')
print('Guardado: acciones_preparadas.csv —', len(corpus), 'partidos.')